<img src="https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/IMPERIAL_logo_RGB_Blue_2024.svg" alt="Imperial Logo" width="500"/><br /><br />

Ensemble Methods and Unsupervised Learning
==============
### SURG70098 - Surgical Data Science and AI
### Stuart Bowyer

## How to use this notebook

Session 6 ended with three models close together on the same patients: logistic regression, a tuned neural network, and logistic regression given the distance of each vital sign from normal. The decision tree was the worst of the groups' five models. Today we combine many trees into one model, deal with an outcome that only one stay in nine has, and look at learning without an outcome at all. Four kinds of activity are marked with **»**:

* **» Predict** — *before* running, decide what the output will be, then run to check. Only used where you have already met the code.
* **» Investigate** — run a working cell, then answer a question about it, from the output or by changing one thing.
* **» Modify** — change a working cell so it answers the next question.
* **» Make** — write the code yourself. The pattern is always somewhere above. **Hint** blocks fold open if you need them: try first.

Part way through, an **Independent practice** slide hands over to you for a long stretch of self-paced work; a **Regroup** slide marks where we come back together.

The notebook is **one environment**: the imports, the tables and the split exist in every cell after them, so run the cells in order. After a restart, or if a cell complains that `X_train` is not defined, use **Runtime → Run before** to catch up. Every practice cell needs only the setup in section 1.

**MIMIC-IV outputs are not in the published slides or notes** beyond summary figures. The data use agreement forbids sharing the data, so you see the rest only when you run the cells yourself. For the same reason, never paste MIMIC rows into an AI assistant or share them; asking about your code is fine.

> Worked solutions, with explanations, are in the solutions notebook, released after the session.

## Welcome

**Goal:** by the end of today you will be able to build a random forest and a gradient-boosted model, judge them against logistic regression, and handle an outcome that few patients have.

It starts with two talks, then picks up where Session 6 left off.

### Before we start
1. Open this notebook in Colab: [Open in Colab](https://colab.research.google.com/github/stuartbowyer/sdsai-lecture-notes/blob/main/notebooks/07_Ensemble_Methods_and_Unsupervised_Learning.ipynb).
2. **File → Save a copy in Drive**, and work in your copy.
3. Run the **setup** cells in section 1 now, so they are ready when the talks end.

## Intended Learning Outcomes
By the end of this session you will be able to:

1. **Explain** how bagging and boosting combine many decision trees, and why each one beats a single tree.
2. **Tune** a random forest and a gradient-boosted model by cross-validation, and **compare** them with logistic regression fold by fold.
3. **Explain** what resampling an imbalanced outcome does to a model's AUC and to its predicted risks, and keep it out of the validation folds.
4. **Describe** clustering and principal component analysis, and **apply** k-means to scaled clinical features.

## Session outline
1. Setup, and two talks: causal inference, and software as a medical device
2. From one tree to a forest
3. Boosting
4. Class imbalance
5. Unsupervised learning
6. Independent practice: forests and boosting for in-hospital death
7. Closing

# 1. Setup
The same course views, table and patient split as Sessions 5 and 6. The first time, Colab asks you to sign in: use the **Google account linked to PhysioNet**.

**Run this once.**

In [ ]:
%pip install pandas_gbq --quiet
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas_gbq
import sklearn

# scikit-learn steps return tables with their column names, not bare arrays
sklearn.set_config(transform_output='pandas')

# The Google Cloud project that pays for your queries. After the course, use your own.
PROJECT = 'bowyerlab-mimic-teaching'

def query(sql, project=PROJECT):
    """Run a SQL query on BigQuery and return the result as a DataFrame."""
    return pandas_gbq.read_gbq(sql, project_id=project)

# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.patients
df_patients = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.patients`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.admissions
df_admissions = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.admissions`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_icu.icustays
df_icustays = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.icustays`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_derived.first_day_vitalsign
df_vitals = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.first_day_vitalsign`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.diagnoses_icd
df_diagnoses = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.diagnoses_icd`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_derived.first_day_lab
df_labs = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.first_day_lab`")

print(len(df_patients), len(df_admissions), len(df_icustays), len(df_vitals), len(df_diagnoses), len(df_labs))

`df_stays`: one row per ICU stay, as in Session 5.

In [ ]:
df_stays = df_icustays.merge(df_admissions, on=['subject_id', 'hadm_id'])
df_stays = df_stays.merge(df_patients, on='subject_id')
df_stays = df_stays.merge(df_vitals, on=['subject_id', 'stay_id'])
df_stays = df_stays.merge(df_labs[['stay_id', 'hemoglobin_min', 'hematocrit_min']], on='stay_id', how='left')

df_stays['died'] = df_stays['hospital_expire_flag'].astype(int)
age = df_stays['anchor_age'] + df_stays['admittime'].dt.year - df_stays['anchor_year']
df_stays['age'] = age.astype(int)
time_to_death = df_stays['deathtime'] - df_stays['intime']
df_stays['dead_by_24h'] = time_to_death.dt.total_seconds() < 24 * 60 * 60

# These columns arrive as Decimal objects (BigQuery NUMERIC), which pandas cannot average: make them floats
for column in ['temperature_min', 'temperature_max', 'temperature_mean', 'hemoglobin_min', 'hematocrit_min', 'los']:
    df_stays[column] = df_stays[column].astype(float)

print(df_stays.shape)

`df_model`: the first ICU stay of each admission, with admission type and race grouped and one-hot encoded, as in Session 5.

In [ ]:
# The first ICU stay of each admission. Sorted by stay_id so the split below is the same for everyone
df_model = df_stays.sort_values(['intime', 'stay_id'])
df_model = df_model.drop_duplicates('hadm_id', keep='first')
df_model = df_model.sort_values('stay_id')

grouping = {
    'EW EMER.': 'EMERGENCY', 'DIRECT EMER.': 'EMERGENCY',
    'URGENT': 'URGENT',
    'ELECTIVE': 'ELECTIVE', 'SURGICAL SAME DAY ADMISSION': 'ELECTIVE',
    'OBSERVATION ADMIT': 'OBSERVATION', 'EU OBSERVATION': 'OBSERVATION',
    'DIRECT OBSERVATION': 'OBSERVATION', 'AMBULATORY OBSERVATION': 'OBSERVATION',
}
admission_group = df_model['admission_type'].replace(grouping)
adm_dummies = pd.get_dummies(admission_group, prefix='adm').astype(int)

race_group = df_model['race'].str.replace(r' -.*$', '', regex=True)
race_group = race_group.str.replace(' OR ', '/')
race_group = race_group.replace(['UNKNOWN', 'UNABLE TO OBTAIN', 'PATIENT DECLINED TO ANSWER', 'OTHER'], 'UNKNOWN/OTHER')
race_dummies = pd.get_dummies(race_group, prefix='race').astype(int)

df_model = pd.concat([df_model, adm_dummies, race_dummies], axis=1)
print(df_model.shape)

`df_features` and `features`: Session 5's table and its 25 model columns. Imputation and scaling learn from rows, so they come after the split.

In [ ]:
# Clinical limits are fixed numbers, not learnt from the rows, so they can come before the split
df_model['glucose_max'] = df_model['glucose_max'].mask(df_model['glucose_max'] > 2000)
df_model['heart_rate_max'] = df_model['heart_rate_max'].mask(df_model['heart_rate_max'] > 250)

temperature_gap = df_model['temperature_mean'].isna()
df_model['temperature_missing'] = temperature_gap.astype(int)

is_icd9_diabetes = df_diagnoses['icd_code'].str.startswith('250')
is_icd10_diabetes = df_diagnoses['icd_code'].str.startswith('E10') | df_diagnoses['icd_code'].str.startswith('E11')
df_diabetes = df_diagnoses[is_icd9_diabetes | is_icd10_diabetes]
has_code = df_model['hadm_id'].isin(df_diabetes['hadm_id'])
df_model['diabetes'] = has_code.astype(int)

numeric_features = ['age', 'heart_rate_mean', 'heart_rate_max', 'resp_rate_mean', 'sbp_min', 'glucose_max']
binary_features = ['temperature_missing', 'diabetes'] + list(adm_dummies.columns) + list(race_dummies.columns)
other_columns = ['subject_id', 'anchor_year_group', 'died', 'los', 'dead_by_24h', 'hemoglobin_min', 'hematocrit_min', 'temperature_mean']

# The model's columns: Session 4's, without diabetes, which is only coded after discharge
kept_binary = binary_features.copy()
kept_binary.remove('diabetes')
features = numeric_features + ['temperature_mean'] + kept_binary

df_features = df_model[other_columns + numeric_features + binary_features].copy()
df_features = df_features.dropna(subset=numeric_features)
print(df_features.shape)
print('Stays that ended in death:', round(df_features['died'].mean(), 3))

`df_train` and `df_test`: Session 5's split of the **patients**, 80% to training and 20% to test, with the labels `y_train` and `y_test`.

In [ ]:
from sklearn.model_selection import train_test_split

patients = df_features['subject_id'].unique()
train_patients, test_patients = train_test_split(patients, test_size=0.2, random_state=123)

in_train = df_features['subject_id'].isin(train_patients)
in_test = df_features['subject_id'].isin(test_patients)
df_train = df_features[in_train].copy()
df_test = df_features[in_test].copy()

y_train = df_train['died']
y_test = df_test['died']
print(len(df_train), 'training rows,', len(df_test), 'test rows')

`X_train` and `X_test`: the features imputed and scaled by a pipeline fitted on the training rows only, as in Session 5 section 4.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

prepare = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
prepare.fit(df_train[features])

X_train = prepare.transform(df_train[features])
X_test = prepare.transform(df_test[features])

## Talks: causal inference, and software as a medical device
Five minutes from each of this week's groups.

**While they talk, write down one question you would ask them.** Afterwards, two or three of you will ask yours.

# 2. From one tree to a forest
In Session 6, the groups' decision tree was the weakest of their five models, at 0.756 on the test patients, against 0.80 for logistic regression and 0.81 for the tuned network. Today: **can many trees together beat one?**

## One tree, no limits
First, one tree with no limits on its size. It keeps splitting until every leaf holds patients with the same outcome.

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=0)
tree.fit(X_train, y_train)

# The leaf each training row ends in, and how many rows share each leaf
leaf = tree.apply(X_train)
rows_per_leaf = pd.Series(leaf).value_counts()

prob_tree_train = tree.predict_proba(X_train)[:, 1]
prob_tree = tree.predict_proba(X_test)[:, 1]
print('Depth:', tree.get_depth(), ' leaves:', tree.get_n_leaves(), ' training rows:', len(X_train))
print('Leaves holding a single training row:', (rows_per_leaf == 1).sum())
print('AUC, training rows:', round(roc_auc_score(y_train, prob_tree_train), 3))
print('AUC, test rows:    ', round(roc_auc_score(y_test, prob_tree), 3))

**» Investigate** — **Which of Session 4's Models A, B and C is this tree?** What has it learned about the training patients that does not hold for new ones?

## Would it grow the same tree again?
306 of the tree's leaves hold a single patient. Each one only records what happened to that patient, not a rule that holds for anyone else. Grow the tree on a slightly different set of patients, and those leaves would change. Would the rest of the tree change too?

We have no other patients, but we can make slightly different sets from these. A **bootstrap sample** draws training rows at random, **with replacement**, as many as there are: some patients appear twice, and about a third not at all.

**» Predict** — This cell grows the same deep tree on ten bootstrap samples. **How similar do you think the ten trees will be?**

In [ ]:
rng = np.random.default_rng(0)
flags = []
trees = []

for sample in range(10):
    rows = rng.integers(0, len(X_train), len(X_train))
    X_sample = X_train.iloc[rows]
    y_sample = y_train.iloc[rows]

    tree_sample = DecisionTreeClassifier(random_state=0)
    tree_sample.fit(X_sample, y_sample)

    # Its first split, and who it flags at 0.5
    first_feature = tree_sample.tree_.feature[0]
    first_split = X_train.columns[first_feature]
    prob_sample = tree_sample.predict_proba(X_test)[:, 1]
    flagged = prob_sample >= 0.5
    flags.append(flagged)
    auc = roc_auc_score(y_test, prob_sample)
    row = {
        'first split': first_split,
        'leaves': tree_sample.get_n_leaves(),
        'flags': flagged.sum(),
        'test AUC': round(auc, 3),
    }
    trees.append(row)

pd.DataFrame(trees)

**Who do the trees flag?** Each flags about 200 test patients. Let's see whether they are the same 200.

In [ ]:
# One row per test patient, one column per tree
by_patient = pd.DataFrame(flags).T
votes = by_patient.sum(axis=1)

flagged_any = votes[votes > 0]
how_many = pd.cut(flagged_any, bins=[0, 1, 5, 10],
                  labels=['1 tree', '2 to 5 trees', '6 to 10 trees'])
print('Flagged by at least one tree:', len(flagged_any), 'of', len(votes))
print(how_many.value_counts(sort=False))

died = y_test.to_numpy() == 1
deaths_per_tree = by_patient[died].sum()
caught_any = (votes[died] > 0).sum()
print('Deaths:', died.sum())
print('Deaths flagged by each tree, on average:', round(deaths_per_tree.mean()))
print('Deaths flagged by at least one tree:', caught_any)

**» Investigate** — Each tree catches about a third of the deaths, but not the same third. **What could you do with ten trees like these?**

## Bagging: bootstrap aggregating
*   Deep trees trained on almost the same patients disagree: their predictions have high **variance**. Averaging many cancels the disagreement and keeps what they share
*   **Bootstrap:** grow many models, each on its own bootstrap sample of the training rows, independently of the others
*   **Aggregate:** combine the trees' predictions: a majority vote, or, as scikit-learn does, the average of their risks. The average gives a risk, so you can rank patients and choose a threshold

<img src="https://upload.wikimedia.org/wikipedia/commons/c/c8/Ensemble_Bagging.svg" alt="Bagging: models trained in parallel on bootstrap samples, then combined" style="height: 280px; display: block; margin: 0 auto;">

By <a href="//commons.wikimedia.org/wiki/User:Sirakorn" title="User:Sirakorn">Sirakorn</a> - <span class="int-own-work" lang="en">Own work</span>, <a href="https://creativecommons.org/licenses/by-sa/4.0" title="Creative Commons Attribution-Share Alike 4.0">CC BY-SA 4.0</a>, <a href="https://commons.wikimedia.org/w/index.php?curid=85888768">Link</a>

## Why averaging helps a deep tree
*   A deep tree's leaves are pure, so its risk is always 0 or 1: it cannot rank patients, only say yes or no
*   So the average is the share of trees that say "died". Ten trees give eleven levels of risk; 300, hundreds. Now patients can be ranked
*   On the ROC curve, one tree is a single cut-off; the average of ten has a point for each level

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

# Each deep tree's risk is 0 or 1, so the ten trees' flags are their risks
risk_ten = by_patient.mean(axis=1)

fig, ax = plt.subplots(figsize=(5.2, 4.6))
lines = [('One deep tree', prob_tree, '#c0392b'), ('Average of ten trees', risk_ten, '#0000cd')]
for name, risk, colour in lines:
    fpr, tpr, cutoffs = roc_curve(y_test, risk)
    # Published: leave out any cut-off that flags fewer than 10 test patients
    flagged = np.array([(risk >= cutoff).sum() for cutoff in cutoffs])
    keep = (flagged == 0) | (flagged >= 10)
    auc = roc_auc_score(y_test, risk)
    ax.plot(fpr[keep], tpr[keep], marker='o', markersize=4, color=colour, label=f'{name}: AUC {auc:.2f}')
ax.plot([0, 1], [0, 1], color='#999999', linestyle='--', linewidth=1)
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate (sensitivity)')
ax.legend(frameon=False, loc='lower right')
for side in ['top', 'right']:
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

## Random forests
*   Bagging alone still grows similar trees: the strongest feature wins at the top of most of them. Of 300 bagged trees, 213 split first on `resp_rate_mean`, and trees that look alike make the same mistakes
*   A **random forest** adds one rule: at each split, a tree may only choose from a **random handful of the features**, by default √25 = 5, drawn afresh at every split
*   The strongest feature is often not on offer, so the trees are pushed apart: 18 different first splits instead of 3
*   `RandomForestClassifier`: `n_estimators` trees, each grown deep by default; `max_features` sets the size of the handful


<img src="https://upload.wikimedia.org/wikipedia/commons/7/76/Random_Forest_Diagram_Extra_Wide.png" alt="A random forest: many trees vote on the class" style="height: 190px; display: block; margin: 0 auto;">

By <a href="//commons.wikimedia.org/w/index.php?title=User:CollaborativeGeneticist&amp;action=edit&amp;redlink=1" class="new" title="User:CollaborativeGeneticist (page does not exist)">CollaborativeGeneticist</a> - <span class="int-own-work" lang="en">Own work</span>, <a href="https://creativecommons.org/licenses/by-sa/4.0" title="Creative Commons Attribution-Share Alike 4.0">CC BY-SA 4.0</a>, <a href="https://commons.wikimedia.org/w/index.php?curid=113209159">Link</a>

**» Predict** — Each deep tree scored about 0.60 on the test patients. **Write down the test AUC you expect from a forest of 300 of them.**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=300, random_state=0)
forest.fit(X_train, y_train)

prob_forest_train = forest.predict_proba(X_train)[:, 1]
prob_forest = forest.predict_proba(X_test)[:, 1]
print('AUC, training rows:', round(roc_auc_score(y_train, prob_forest_train), 3))
print('AUC, test rows:    ', round(roc_auc_score(y_test, prob_forest), 3))

**» Investigate** — **Change `n_estimators` to 1, 10 and 50.** How many trees does the forest need before it stops improving? From 10 trees the training AUC is 1.00, as it was for the single tree: **is this forest overfitting? How would you tell?**

# 3. Boosting
A forest's trees are grown independently: none knows what the others got wrong. Of the 197 deaths in the test rows, 38 were flagged by none of the ten trees.

**» Investigate** — **Could averaging more trees like these ever flag those 38? If not, how would you build the next tree differently?**

## Boosting, one step at a time
Session 6's dots: the share who died in each tenth of the patients by temperature. Boosting on temperature alone:

*   Start every patient at the training death rate, 11%: the flat line
*   Each patient's **error** is their outcome minus their current risk: 1 − 0.11 = +0.89 for a patient who died, 0 − 0.11 = −0.11 for one who survived
*   A small tree learns to predict the error. A tree can predict a number as well as a class: each leaf predicts the average error of its patients, which is the death rate in the leaf minus the current risk
*   Here the tree has three leaves, three ranges of temperature, and moves each range up or down

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

# Session 6's dots: the share who died in each tenth of the patients by temperature, hundreds each
with_temperature = df_features.dropna(subset=['temperature_mean'])
tenth = pd.qcut(with_temperature['temperature_mean'], 10)
by_tenth = with_temperature.groupby(tenth, observed=True).agg(
    temperature=('temperature_mean', 'mean'), died=('died', 'mean'), patients=('died', 'size'))

temperature_train = df_train.dropna(subset=['temperature_mean'])
temperatures = pd.DataFrame({'temperature_mean': np.linspace(35.5, 38.5, 300)})

# Small trees of three leaves, on temperature alone; every leaf holds at least 20 training patients
boost_u = HistGradientBoostingClassifier(max_leaf_nodes=3, learning_rate=0.1, max_iter=100, random_state=0)
boost_u.fit(temperature_train[['temperature_mean']], temperature_train['died'])
stages = [prob[:, 1] for prob in boost_u.staged_predict_proba(temperatures)]
start = np.full(len(temperatures), temperature_train['died'].mean())


def draw_boosting(shown):
    """The risk curve after each number of trees in `shown`, over Session 6's dots."""
    fig, axes = plt.subplots(1, len(shown), figsize=(4 * len(shown), 3.2), sharey=True)
    for ax, trees in zip(axes, shown):
        curve = start if trees == 0 else stages[trees - 1]
        ax.scatter(by_tenth['temperature'], by_tenth['died'], color='#444444', s=18, zorder=3)
        ax.plot(temperatures['temperature_mean'], curve, color='#c0392b', linewidth=2)
        ax.set_title('1 tree' if trees == 1 else f'{trees} trees', fontsize=10)
        ax.set_xlabel('Mean temperature (°C)')
        ax.set_ylim(0, 0.45)
        for side in ['top', 'right']:
            ax.spines[side].set_visible(False)
    axes[0].set_ylabel('Risk of death')
    plt.tight_layout()
    plt.show()


draw_boosting([0, 1])

## Adding trees
*   Each new tree is fitted to the errors that remain, and the **learning rate**, 0.1 here, shrinks every move
*   After 20 trees the steps make the U. After 100, the line follows the training patients' quirks: unlike a forest, more trees can overfit

In [ ]:
draw_boosting([5, 20, 100])

## Boosting: small trees in sequence
1. Every patient starts at the training death rate
2. Each small tree learns the errors that remain, and moves each patient's risk by its leaf's value, shrunk by the learning rate
3. A patient's risk is the starting risk plus every tree's moves. The moves add up inside a sigmoid, as the weighted sum does in logistic regression, so the risk stays between 0 and 1

*   Each small tree is too simple, and makes the same mistakes whatever patients it learns from: high **bias**. Adding them up, each correcting the last, reduces it
*   Unlike a forest, more trees is not always better: enough of them will fit the training patients' noise

## Gradient boosting in scikit-learn
*   `HistGradientBoostingClassifier`: groups each feature's values into bins first, so it is fast on large tables, and it accepts missing values without an imputer
*   Its main settings: `learning_rate` (default 0.1), `max_iter`, the number of trees (100), and `max_leaf_nodes`, the size of each tree (31)
*   `GradientBoostingClassifier` is the older, slower version of the same method
*   **XGBoost** and **LightGBM** are separate libraries for the same idea, with extra regularisation, and are widely used. They have the same `fit` and `predict_proba`

Gradient boosting on the same rows, at the default learning rate.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

# How far each new tree moves the prediction
rate = 0.1

boosting = HistGradientBoostingClassifier(learning_rate=rate, random_state=0)
boosting.fit(X_train, y_train)

prob_boost_train = boosting.predict_proba(X_train)[:, 1]
prob_boost = boosting.predict_proba(X_test)[:, 1]
print('AUC, training rows:', round(roc_auc_score(y_train, prob_boost_train), 3))
print('AUC, test rows:    ', round(roc_auc_score(y_test, prob_boost), 3))

**» Investigate** — **Change `rate` to 0.01, 0.03, 0.3 and 1.** Which does best on the test patients? What goes wrong at each end? (Choosing on the test rows is for this demonstration only: Part 2 of the practice chooses by cross-validation.)

## Bagging and boosting

| | Bagging: random forest | Boosting: gradient boosting |
| :-- | :-- | :-- |
| Trees | Deep, each overfits | Small, each underfits |
| Built | Independently, in parallel | One after another, each correcting the last |
| Reduces | Variance | Bias |
| Main settings | `n_estimators`, `min_samples_leaf`, `max_features` | `learning_rate`, `max_iter`, `max_leaf_nodes` |
| More trees | Never worse, only slower | Eventually overfits |

## Why trees suit tables
*   **No scaling:** a split asks "is glucose above 180?", so skewed values, outliers and units do not matter
*   **Shapes for free:** thresholds, U-shapes and interactions come from splitting more than once. Logistic regression needs them built by hand, like Session 6's distances from normal
*   **Missing values:** boosting learns which way to send them. In health records, a test that was not ordered can carry information
*   **Useless columns are ignored:** a split never chooses them, while a network spreads weight across every input ([Grinsztajn et al., 2022](https://arxiv.org/abs/2207.08815))

## But on small clinical data
*   Flexibility needs data: random forests and networks needed over ten times as many events per variable as logistic regression before their AUC on new patients stopped improving and matched their AUC on the training patients ([van der Ploeg et al., 2014](https://doi.org/10.1186/1471-2288-14-137))
*   Across 71 clinical prediction studies, the carefully done comparisons found no AUC advantage over logistic regression ([Christodoulou et al., 2019](https://doi.org/10.1016/j.jclinepi.2019.02.004))
*   So: logistic regression is the baseline. A forest or boosting earns its place by beating it fold by fold, and you still have to check its calibration

# 4. Class imbalance
Adverse outcomes are usually rare. In our training rows, about one stay in nine ended in death: every model so far learned from that mix.

In [ ]:
deaths = y_train.value_counts()
print(deaths)
print('Share who died:', round(y_train.mean(), 3))

### A model that predicts "survived" for everyone is right 89% of the time. How would you make a model pay more attention to the deaths?

**» Predict** — `RandomOverSampler`, from the `imbalanced-learn` library, copies the training deaths until they match the survivors. Its own pipeline does this inside `fit`, so only training rows are copied. **Write down what you expect to happen to the test AUC, and to the average predicted risk.**

In [ ]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import make_pipeline as make_imb_pipeline
from sklearn.linear_model import LogisticRegression

pipeline_plain = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                               LogisticRegression())
pipeline_plain.fit(df_train[features], y_train)
pipeline_over = make_imb_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                                  RandomOverSampler(random_state=1), LogisticRegression())
pipeline_over.fit(df_train[features], y_train)

prob_plain = pipeline_plain.predict_proba(df_test[features])[:, 1]
prob_over = pipeline_over.predict_proba(df_test[features])[:, 1]
print('Test AUC, as it is:   ', round(roc_auc_score(y_test, prob_plain), 3))
print('Test AUC, oversampled:', round(roc_auc_score(y_test, prob_over), 3))
print('Average predicted risk of death, as it is:   ', round(prob_plain.mean(), 3))
print('Average predicted risk of death, oversampled:', round(prob_over.mean(), 3))
print('Share who actually died, test rows:          ', round(y_test.mean(), 3))

## Resampling moves the threshold
*   The AUC measures **ranking** (Session 5). Copying the deaths pushes every patient's risk up, but hardly changes their order
*   At the usual cut-off of 0.5, the oversampled model flags far more patients. The unchanged model does the same if you lower its threshold to the share who died
*   The cost: the oversampled model's risks no longer mean what they say. On average it predicts about 40%, for patients of whom about 12% died

In [ ]:
from sklearn.metrics import recall_score

# The unchanged model's threshold: the training death rate
threshold = y_train.mean()

flagged_over = prob_over >= 0.5
flagged_plain = prob_plain >= threshold
print('Oversampled, at 0.5:   flagged', round(flagged_over.mean(), 3),
      ' sensitivity', round(recall_score(y_test, flagged_over), 2))
print('Unchanged, at', round(threshold, 3), ': flagged', round(flagged_plain.mean(), 3),
      ' sensitivity', round(recall_score(y_test, flagged_plain), 2))

**» Predict** — This cell makes a mistake on purpose: it oversamples the **whole** training table first, then cross-validates a random forest with ordinary shuffled folds. Copies of the same patient can now be in both the training and the validation folds. **Write down the cross-validated AUC you expect.**

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

# The mistake: resampling before the folds are made
oversampler = RandomOverSampler(random_state=1)
X_copied, y_copied = oversampler.fit_resample(df_train[features], y_train)

forest_100 = RandomForestClassifier(n_estimators=100, random_state=0)
pipeline_forest = make_pipeline(SimpleImputer(strategy='median'), forest_100)
shuffled_folds = KFold(n_splits=5, shuffle=True, random_state=0)
scores_leaky = cross_val_score(pipeline_forest, X_copied, y_copied,
                               cv=shuffled_folds, scoring='roc_auc')
print('Oversampled first, mean AUC:', round(scores_leaky.mean(), 3))

**» Investigate** — The same forest, with the oversampler **inside** the pipeline, so only each training fold is copied, and folds made by patient as in Sessions 5 and 6. **Where did the difference go?** Which part of Session 5's validation would have caught it?

In [ ]:
from sklearn.model_selection import GroupKFold

pipeline_forest_over = make_imb_pipeline(
    SimpleImputer(strategy='median'),
    RandomOverSampler(random_state=1),
    RandomForestClassifier(n_estimators=100, random_state=0),
)
scores_honest = cross_val_score(
    pipeline_forest_over,
    df_train[features],
    y_train,
    cv=GroupKFold(n_splits=5),
    groups=df_train['subject_id'],
    scoring='roc_auc',
)
print('Oversampled inside each fold, mean AUC:', round(scores_honest.mean(), 3))

## What to do about imbalance
*   **First, nothing.** Train on the data as they are; judge the model by its AUC and its calibration, which resampling does not improve
*   **Choose the threshold** for the decision, from the cost of a missed patient against a false alarm (Session 5)
*   If a model needs it, `class_weight='balanced'` weights the rarer class in training without copying rows. It moves the risks in the same way as oversampling
*   Resampling or `class_weight` earn their place when the outcome is so rare that a model barely learns it: a handful of events, or a network trained in small batches. Recalibrate afterwards, so the risks mean what they say
*   If you do resample, resample **inside** the pipeline, so validation rows are never copied
*   An AUC near 1.00 on a clinical table is a bug until proven otherwise

# 5. Unsupervised learning
*   **What:** learning from the features alone, with no outcome: no `died`, no `y_train`
*   **Why:** to find structure in the patients that nobody has labelled
*   **Methods:**
    *   **Clustering:** groups of patients with similar features
    *   **Dimensionality reduction:** fewer columns that keep most of the information

## Clustering
*   **What:** groups of patients whose features are close together
*   **Why:** to reveal structure that simple summaries hide
*   **Uses:**
    *   Identifying phenotypes of complex disease subtypes (e.g. in cardiac disease "young-low comorbidity burden; metabolic; cardio-renal; etc," - [link.springer.com/article/10.1007/s11897-023-00615-z](https://link.springer.com/article/10.1007/s11897-023-00615-z))
    *   Segmenting structures from medical images
    *   Grouping patients to predict prognosis or recovery time

### Clustering Methods
*   **Centroid-Based Methods:** assign clusters based on proximity to a given centroid (e.g. K-Means)
*   **Hierarchical Methods:** build clusters in a hierarchy by recursively either combining (e.g. agglomerative) or splitting (e.g. divisive) clusters
*   **Density-Based Methods:** identify clusters based on contiguous areas of high-density data surrounded by low density data (e.g. DBSCAN)

In [ ]:
# This cell's code uses plain arrays, not tables
sklearn.set_config(transform_output='default')

# Taken from: https://scikit-learn.org/1.5/auto_examples/cluster/plot_cluster_comparison.html

import time
import warnings
from itertools import cycle, islice

import matplotlib.pyplot as plt
import numpy as np

from sklearn import cluster, datasets, mixture
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import StandardScaler

# ============
# Generate datasets. We choose the size big enough to see the scalability
# of the algorithms, but not too big to avoid too long running times
# ============
n_samples = 500
seed = 30
noisy_circles = datasets.make_circles(
    n_samples=n_samples, factor=0.5, noise=0.05, random_state=seed
)
noisy_moons = datasets.make_moons(n_samples=n_samples, noise=0.05, random_state=seed)
blobs = datasets.make_blobs(n_samples=n_samples, random_state=seed)
rng = np.random.RandomState(seed)
no_structure = rng.rand(n_samples, 2), None

# Anisotropicly distributed data
random_state = 170
X, y = datasets.make_blobs(n_samples=n_samples, random_state=random_state)
transformation = [[0.6, -0.6], [-0.4, 0.8]]
X_aniso = np.dot(X, transformation)
aniso = (X_aniso, y)

# blobs with varied variances
varied = datasets.make_blobs(
    n_samples=n_samples, cluster_std=[1.0, 2.5, 0.5], random_state=random_state
)

# ============
# Set up cluster parameters
# ============
plt.figure(figsize=(9 * 2 + 3, 6))
plt.subplots_adjust(
    left=0.02, right=0.98, bottom=0.001, top=0.95, wspace=0.05, hspace=0.01
)

plot_num = 1

default_base = {
    "quantile": 0.3,
    "eps": 0.3,
    "damping": 0.9,
    "preference": -200,
    "n_neighbors": 3,
    "n_clusters": 3,
    "min_samples": 7,
    "xi": 0.05,
    "min_cluster_size": 0.1,
    "allow_single_cluster": True,
    "hdbscan_min_cluster_size": 15,
    "hdbscan_min_samples": 3,
    "random_state": 42,
}

datasets = [
    (
        noisy_circles,
        {
            "damping": 0.77,
            "preference": -240,
            "quantile": 0.2,
            "n_clusters": 2,
            "min_samples": 7,
            "xi": 0.08,
        },
    ),
    (
        noisy_moons,
        {
            "damping": 0.75,
            "preference": -220,
            "n_clusters": 2,
            "min_samples": 7,
            "xi": 0.1,
        },
    ),
    (
        varied,
        {
            "eps": 0.18,
            "n_neighbors": 2,
            "min_samples": 7,
            "xi": 0.01,
            "min_cluster_size": 0.2,
        },
    ),
    (
        aniso,
        {
            "eps": 0.15,
            "n_neighbors": 2,
            "min_samples": 7,
            "xi": 0.1,
            "min_cluster_size": 0.2,
        },
    ),
    (blobs, {"min_samples": 7, "xi": 0.1, "min_cluster_size": 0.2}),
    (no_structure, {}),
]

for i_dataset, (dataset, algo_params) in enumerate(datasets):
    # update parameters with dataset-specific values
    params = default_base.copy()
    params.update(algo_params)

    X, y = dataset

    # normalize dataset for easier parameter selection
    X = StandardScaler().fit_transform(X)

    # estimate bandwidth for mean shift
    bandwidth = cluster.estimate_bandwidth(X, quantile=params["quantile"])

    # connectivity matrix for structured Ward
    connectivity = kneighbors_graph(
        X, n_neighbors=params["n_neighbors"], include_self=False
    )
    # make connectivity symmetric
    connectivity = 0.5 * (connectivity + connectivity.T)

    # ============
    # Create cluster objects
    # ============
    ms = cluster.MeanShift(bandwidth=bandwidth, bin_seeding=True)
    two_means = cluster.MiniBatchKMeans(
        n_clusters=params["n_clusters"],
        random_state=params["random_state"],
    )
    ward = cluster.AgglomerativeClustering(
        n_clusters=params["n_clusters"], linkage="ward", connectivity=connectivity
    )
    spectral = cluster.SpectralClustering(
        n_clusters=params["n_clusters"],
        eigen_solver="arpack",
        affinity="nearest_neighbors",
        random_state=params["random_state"],
    )
    dbscan = cluster.DBSCAN(eps=params["eps"])
    hdbscan = cluster.HDBSCAN(
        min_samples=params["hdbscan_min_samples"],
        min_cluster_size=params["hdbscan_min_cluster_size"],
        allow_single_cluster=params["allow_single_cluster"],
    )
    optics = cluster.OPTICS(
        min_samples=params["min_samples"],
        xi=params["xi"],
        min_cluster_size=params["min_cluster_size"],
    )
    affinity_propagation = cluster.AffinityPropagation(
        damping=params["damping"],
        preference=params["preference"],
        random_state=params["random_state"],
    )
    average_linkage = cluster.AgglomerativeClustering(
        linkage="average",
        metric="cityblock",
        n_clusters=params["n_clusters"],
        connectivity=connectivity,
    )
    birch = cluster.Birch(n_clusters=params["n_clusters"])
    gmm = mixture.GaussianMixture(
        n_components=params["n_clusters"],
        covariance_type="full",
        random_state=params["random_state"],
    )

    clustering_algorithms = (
        ("$\\bf{CentroidBased}$\nMiniBatch\nKMeans", two_means),
        ("$\\bf{CentroidBased}$\nMeanShift", ms),
        ("$\\bf{Hierarchical}$\nAgglomerative\nClustering", average_linkage),
        ("$\\bf{Hierarchical}$\nWard", ward),
        ("$\\bf{Hierarchical}$\nBIRCH", birch),
        ("$\\bf{DensityBased}$\nDBSCAN", dbscan),
        ("$\\bf{DensityBased}$\nHDBSCAN", hdbscan),
        ("$\\bf{DensityBased}$\nOPTICS", optics),
        ("$\\bf{GraphBased}$\nAffinity\nPropagation", affinity_propagation),
        ("$\\bf{GraphBased}$\nSpectral\nClustering", spectral),
        ("$\\bf{ModelBased}$\nGaussian\nMixture", gmm),
    )

    for name, algorithm in clustering_algorithms:
        t0 = time.time()

        # catch warnings related to kneighbors_graph
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore",
                message="the number of connected components of the "
                + "connectivity matrix is [0-9]{1,2}"
                + " > 1. Completing it to avoid stopping the tree early.",
                category=UserWarning,
            )
            warnings.filterwarnings(
                "ignore",
                message="Graph is not fully connected, spectral embedding"
                + " may not work as expected.",
                category=UserWarning,
            )
            algorithm.fit(X)

        t1 = time.time()
        if hasattr(algorithm, "labels_"):
            y_pred = algorithm.labels_.astype(int)
        else:
            y_pred = algorithm.predict(X)

        plt.subplot(len(datasets), len(clustering_algorithms), plot_num)
        if i_dataset == 0:
            plt.title(name, size=14)

        colors = np.array(
            list(
                islice(
                    cycle(
                        [
                            "#377eb8",
                            "#ff7f00",
                            "#4daf4a",
                            "#f781bf",
                            "#a65628",
                            "#984ea3",
                            "#999999",
                            "#e41a1c",
                            "#dede00",
                        ]
                    ),
                    int(max(y_pred) + 1),
                )
            )
        )
        # add black color for outliers (if any)
        colors = np.append(colors, ["#000000"])
        plt.scatter(X[:, 0], X[:, 1], s=10, color=colors[y_pred])

        plt.xlim(-2.5, 2.5)
        plt.ylim(-2.5, 2.5)
        plt.xticks(())
        plt.yticks(())
        plot_num += 1

plt.show()
sklearn.set_config(transform_output='pandas')

Modified from [scikit-learn.org/1.5/auto_examples/cluster/plot_cluster_comparison.html](https://scikit-learn.org/1.5/auto_examples/cluster/plot_cluster_comparison.html)

## K-means clustering
*   Choose the number of clusters, *k*. Place *k* **centroids**, cluster centres, at random
*   Assign each patient to the nearest centroid; move each centroid to the mean of its patients; repeat until nothing changes
*   "Nearest" is a **distance** between rows of features, as in this week's first talk

![K-Means](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-06_K-Means-Clustering.svg)

**Which of the shapes in the figure above would k-means get wrong?**

## K-means in scikit-learn
Three clusters from two first-day vital signs. Each row of the table is a cluster: its number of patients, its means, and the share who died, which k-means never saw.

In [ ]:
from sklearn.cluster import KMeans

vitals = df_features[['heart_rate_mean', 'resp_rate_mean']]

kmeans = KMeans(n_clusters=3, random_state=1)
cluster = kmeans.fit_predict(vitals)

df_clusters = df_features.assign(cluster=cluster)
by_cluster = df_clusters.groupby('cluster')
summary = by_cluster.agg(
    patients=('died', 'size'),
    heart_rate=('heart_rate_mean', 'mean'),
    resp_rate=('resp_rate_mean', 'mean'),
    died=('died', 'mean'),
)
summary.round(2)

**» Investigate** — **What has gone wrong here?** Compare how far apart the clusters are in heart rate and in respiratory rate. Why did k-means use one more than the other? Session 4 met the same problem.

## Scale first
The same clusters, with both vital signs scaled first.

In [ ]:
scaler = StandardScaler()
scaled_vitals = scaler.fit_transform(vitals)

kmeans_scaled = KMeans(n_clusters=3, random_state=1)
cluster_scaled = kmeans_scaled.fit_predict(scaled_vitals)

df_clusters = df_features.assign(cluster=cluster_scaled)
by_cluster = df_clusters.groupby('cluster')
summary = by_cluster.agg(
    patients=('died', 'size'),
    heart_rate=('heart_rate_mean', 'mean'),
    resp_rate=('resp_rate_mean', 'mean'),
    died=('died', 'mean'),
)
summary.round(2)

**» Investigate** — **Describe each cluster as a clinician might.** k-means never saw `died`: do the clusters still differ in risk? What would you need before calling one of them a phenotype?

## Dimensionality reduction
*   **What:** fewer columns that keep most of the information in many
*   **Why:** columns that measure the same thing add little; with many columns and few patients, models overfit, and distances mean less
*   **Uses:**
    *   Simplifying genomic data - [journals.plos.org/plosone/article?id=10.1371%2Fjournal.pone.0093766](https://journals.plos.org/plosone/article?id=10.1371%2Fjournal.pone.0093766)
    *   Simplifying timeseries data - [link.springer.com/article/10.1007/s11227-021-04303-4](https://link.springer.com/article/10.1007/s11227-021-04303-4)

## Feature projection
*   **What:** new features, each a weighted combination of the old ones, fewer of them, that keep most of the information
*   **How:** project the patients onto a lower-dimensional space, as a shadow projects three dimensions onto two

![Feature Projection](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-06_Feature-Projection.svg)

## Principal component analysis (PCA)
*   The first **principal component** is the direction along which the patients vary most; the second, the most at right angles to it; and so on
*   Each component is a weighted combination of the features: its **loadings**
*   `explained_variance_ratio_`: the share of the variation each one holds
*   Scale the features first, or the column with the largest numbers dominates

In [ ]:
# This cell's code uses plain arrays, not tables
sklearn.set_config(transform_output='default')

# This code is just for the explanatory plot, you do not need to understand it
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from matplotlib.gridspec import GridSpec

# Generate synthetic 3D data
X, y = make_blobs(n_samples=500, centers=3, n_features=3, random_state=1, cluster_std=2.0)

# Step 1: Original 3D data visualization
fig = plt.figure(figsize=(18, 10))

gs = GridSpec(2, 3, height_ratios=[2, 1])
gs.update(wspace=0.3, hspace=0.05)

ax1 = fig.add_subplot(gs[0,0], projection='3d')
ax1.scatter(X[:, 0], X[:, 1], X[:, 2], c=y)
ax1.set_title("Step 1: Original Feature Space")
ax1.set_xlabel("Feature 1")
ax1.set_ylabel("Feature 2")
ax1.set_zlabel("Feature 3")

# Step 2: Standardise the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

ax2 = fig.add_subplot(gs[0,1], projection='3d')
ax2.scatter(X_scaled[:, 0], X_scaled[:, 1], X_scaled[:, 2], c=y, alpha=0.5)
ax2.set_title("Step 2: Standardise Data")
ax2.set_xlabel("Standardised Feature 1")
ax2.set_ylabel("Standardised Feature 2")
ax2.set_zlabel("Standardised Feature 3")

# Step 3: Compute PCA
pca = PCA(n_components=2, random_state=1)
pca.fit(X_scaled)

# Extract eigenvectors (principal components) and mean
mean = np.mean(X_scaled, axis=0)
components = pca.components_

# Visualize principal components in 3D space
ax3 = fig.add_subplot(gs[0,2], projection='3d')
ax3.scatter(X_scaled[:, 0], X_scaled[:, 1], X_scaled[:, 2], c=y, alpha=0.5)
ax3.scatter(mean[0], mean[1], mean[2], color='red', s=100, label='Mean')
for i, component in enumerate(components):
    ax3.quiver(
        mean[0], mean[1], mean[2], 
        component[0], component[1], component[2],
        color=['blue', 'orange'][i], length=3.0, linewidth=6, label=f'PC {i + 1}'
    )
ax3.set_title("Step 3: Eigen Decomposition & Rank")
ax3.set_xlabel("Standardised Feature 1")
ax3.set_ylabel("Standardised Feature 2")
ax3.set_zlabel("Standardised Feature 3")
ax3.legend()

# Step 4: Project data onto the first two principal components
X_pca = pca.transform(X_scaled)

ax4 = fig.add_subplot(gs[1,1])
ax4.scatter(X_pca[:, 0], X_pca[:, 1], c=y)
# ax4.set_aspect('equal')
ax4.set_title("Step 4: Project Data into Primary Principal Components")
ax4.set_xlabel("Principal Component 1")
ax4.set_ylabel("Principal Component 2")

# Show
# Reduce the vertical space between rows
plt.subplots_adjust(wspace=0.01, hspace=0.01)
plt.show()
sklearn.set_config(transform_output='pandas')

**» Investigate** — Our table has three heart-rate columns: the minimum, mean and maximum over the first day. **How many numbers do you really need to describe a patient's heart rate?** Read the share of variance and the loadings of the first component: what does it measure?

In [ ]:
from sklearn.decomposition import PCA

heart_rates = df_model[['heart_rate_min', 'heart_rate_mean', 'heart_rate_max']]
heart_rates = heart_rates.dropna()
scaler = StandardScaler()
scaled_rates = scaler.fit_transform(heart_rates)

pca = PCA(n_components=3)
pca.fit(scaled_rates)

print('Share of variance, each component:', pca.explained_variance_ratio_.round(3))
print('Loadings of the first component:  ', pca.components_[0].round(2))

# 6. Independent practice: forests and boosting for in-hospital death
From here to the **Regroup** is yours, in your pair: about 60 minutes.

* Each task is marked **» Modify**, **» Make** or **» Investigate**, and most ask for a **comment** recording a decision. Those comments are what we revisit in the Closing.
* **Hint** blocks fold open: try first.
* Stuck? As in Session 6: **copy** the closest cell above, then **change one thing at a time**, and read the last line of any error first.
* Every cell needs only the setup in section 1.

Work together, and call me over when you get stuck. **Understanding beats finishing.**

## Part 1: Settings for a forest
**» Modify** — This cell cross-validates a forest of 300 trees on the training rows. Trees need no scaling, so the pipeline only imputes. **Change `min_samples_leaf` to 5, 20 and 100; then, at the best of those, change `max_features` to 0.5**, half the features at each split. In the comment: **which settings help, and by more than the spread between folds?**

In [ ]:
# Which settings help, and by more than the spread between folds:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline

forest_settings = RandomForestClassifier(n_estimators=300, min_samples_leaf=1, max_features='sqrt', random_state=0)
pipeline_forest = make_pipeline(SimpleImputer(strategy='median'), forest_settings)
scores = cross_val_score(
    pipeline_forest,
    df_train[features],
    df_train['died'],
    cv=GroupKFold(n_splits=5),
    groups=df_train['subject_id'],
    scoring='roc_auc',
)
print('Mean AUC', round(scores.mean(), 3), ' spread between folds', round(scores.std(), 3))

<details><summary><b>What you should find</b></summary>

A minimum of 5 patients per leaf does best, but every setting is within the spread between folds of every other. Leaves of 100 patients make the trees too simple: the bias is back. Letting each split choose from half the features makes the trees more alike, and the forest slightly worse.

</details>

## Part 2: Tune boosting, then test it once
**» Make** — As in Session 6's Part 2, for gradient boosting: **write the grid**: three learning rates, including 0.03, and two numbers of trees, 100 and 300. Then fill in the two blanks below it. In the comment: the chosen values, both AUCs, and is the best value at the edge of the grid?

<details><summary><b>Hint</b></summary>

Session 6's Part 2 has a grid of the same shape: a dictionary with one entry per setting, and a list of values for each. In a pipeline, each setting's name starts with its step's: `'histgradientboostingclassifier__learning_rate'`. Groups are each row's `subject_id`; the test rows are `df_test[features]`. Best value at the edge of the grid? Extend it that way and run again.

</details>

In [ ]:
# Chosen values, CV AUC, test AUC, and is the best value at the edge of the grid:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import make_pipeline

pipeline_boost = make_pipeline(SimpleImputer(strategy='median'), HistGradientBoostingClassifier(random_state=0))

# 1. The grid: in a pipeline, each setting's name starts with its step's
grid = ____

# 2. The search, by cross-validation on the training rows, one patient per fold
search = GridSearchCV(pipeline_boost, grid, cv=GroupKFold(n_splits=5), scoring='roc_auc')
search.fit(df_train[features], df_train['died'], groups=____)
print('Chosen:', search.best_params_, ' CV AUC:', round(search.best_score_, 3))

# 3. Once, at the end: the chosen model on the test rows
prob_search = search.predict_proba(____)[:, 1]
print('Test AUC:', round(roc_auc_score(df_test['died'], prob_search), 3))

## Part 3: Three ways to bend
In Session 6, logistic regression needed the distance of each vital sign from normal to follow risks that rise at both ends; the network found those shapes itself. This cell sets up the comparison: Session 6's `fold_aucs`, and the training table with the five distances.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def fold_aucs(model, table, columns):
    """The AUC on each of the same five folds of the training patients."""
    pipeline = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), model)
    return cross_val_score(pipeline, table[columns], table['died'], cv=GroupKFold(n_splits=5),
                           groups=table['subject_id'], scoring='roc_auc')


# Distance from a normal value, in either direction, as in Session 6
table = df_train.copy()
table['temperature_from_normal'] = (table['temperature_mean'] - 37).abs()
table['heart_rate_mean_from_normal'] = (table['heart_rate_mean'] - 80).abs()
table['sbp_min_from_normal'] = (table['sbp_min'] - 110).abs()
table['resp_rate_mean_from_normal'] = (table['resp_rate_mean'] - 16).abs()
table['glucose_max_from_normal'] = (table['glucose_max'] - 120).abs()
distances = ['temperature_from_normal', 'heart_rate_mean_from_normal', 'sbp_min_from_normal',
             'resp_rate_mean_from_normal', 'glucose_max_from_normal']
features_plus = features + distances

**» Investigate** — Five models on the same five folds. **Put your choices from Parts 1 and 2 into the forest and boosting lines**, then run: about a minute. **Which models beat logistic regression in every fold? Is any of the four ahead of the others by more than the spread between folds?**

In [ ]:
# Which beat logistic regression in every fold, and is any ahead of the others:
# The forest and boosting lines take your settings from Parts 1 and 2
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

aucs_lr = fold_aucs(LogisticRegression(), table, features)
aucs_lr_plus = fold_aucs(LogisticRegression(), table, features_plus)
aucs_net = fold_aucs(MLPClassifier(hidden_layer_sizes=(50,), alpha=1, random_state=0, max_iter=1000), table, features)
aucs_forest = fold_aucs(RandomForestClassifier(n_estimators=300, min_samples_leaf=____, random_state=0), table, features)
aucs_boost = fold_aucs(HistGradientBoostingClassifier(learning_rate=____, max_iter=____, random_state=0), table, features)

print('Logistic regression:        ', aucs_lr.round(3), ' mean', round(aucs_lr.mean(), 3))
print('Logistic regression + dist.:', aucs_lr_plus.round(3), ' mean', round(aucs_lr_plus.mean(), 3))
print('Network:                    ', aucs_net.round(3), ' mean', round(aucs_net.mean(), 3))
print('Forest:                     ', aucs_forest.round(3), ' mean', round(aucs_forest.mean(), 3))
print('Boosting:                   ', aucs_boost.round(3), ' mean', round(aucs_boost.mean(), 3))

**» Modify** — **Give the forest and boosting the distances too:** in their two lines, change `features` to `features_plus`. In a comment: did they gain what logistic regression gained? Why might a tree not need them?

<details><summary><b>Hint</b></summary>

Think about how a tree uses temperature: it can split at 36 °C and again at 38 °C.

</details>

## Part 4: Does oversampling help boosting?
**» Make** — Section 4 showed that oversampling did not help logistic regression. Boosting learns differently, by correcting its errors: does oversampling help it? The cell cross-validates your tuned boosting, as it is, on Part 3's folds. **Below it, cross-validate the same model with `RandomOverSampler` inside the pipeline**, and compare fold by fold. In the comment: does oversampling help in every fold? In any?

<details><summary><b>Hint</b></summary>

Section 4's Predict cell builds a pipeline with `make_imb_pipeline`: the oversampler goes between the imputer and the model. Copy the `cross_val_score` call and change only the pipeline.

</details>

In [ ]:
# Does oversampling help boosting, in every fold or in any:
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import make_pipeline as make_imb_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline

# Your settings from Part 2, as in Part 3
boosting_tuned = HistGradientBoostingClassifier(learning_rate=____, max_iter=____, random_state=0)

pipeline_boost_plain = make_pipeline(SimpleImputer(strategy='median'), boosting_tuned)
aucs_boost_plain = cross_val_score(
    pipeline_boost_plain,
    df_train[features],
    df_train['died'],
    cv=GroupKFold(n_splits=5),
    groups=df_train['subject_id'],
    scoring='roc_auc',
)
print('As it is:   ', aucs_boost_plain.round(3), ' mean', round(aucs_boost_plain.mean(), 3))

# The same model, with the oversampler inside the pipeline

**» Modify (stretch)** — **Weights or a threshold?** This cell fits logistic regression as it is, and reports what a clinician would see at a threshold. **First change `class_weight` to `'balanced'`. Then put it back to `None` and change `threshold` to `y_train.mean()`.** In the comment: **what changed, what did not, and which version would you hand a clinician?**

In [ ]:
# What changed, what did not, and which version you would hand a clinician:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, roc_auc_score

model_weighted = LogisticRegression(class_weight=None)
model_weighted.fit(X_train, y_train)
prob_weighted = model_weighted.predict_proba(X_test)[:, 1]

threshold = 0.5
flagged = prob_weighted >= threshold

print('Test AUC:                       ', round(roc_auc_score(y_test, prob_weighted), 3))
print('Average predicted risk of death:', round(prob_weighted.mean(), 3), ' share who actually died:', round(y_test.mean(), 3))
print('Share flagged:                  ', round(flagged.mean(), 3), ' sensitivity:', round(recall_score(y_test, flagged), 2))

<details><summary><b>What you should find</b></summary>

The AUC does not change. `class_weight='balanced'` raises every risk: the average predicted risk of death goes from about 0.11 to about 0.40, against 0.12 who actually died, and at 0.5 it flags a third of the patients. The unweighted model with its threshold lowered to 0.11 flags the same third, with the same sensitivity, and its risks still mean what they say.

</details>

**» Modify (stretch)** — **Clusters of patients.** The cell clusters the training patients on all seven scaled numeric features. **Change `n_clusters` to 3, 4, 5 and 6.** In the comment: does each cluster have a different death rate, though k-means never saw `died`? Would you trust the clusters from one *k* more than another?

In [ ]:
# Do the clusters differ in death rate, and which k would you trust:
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

numeric = numeric_features + ['temperature_mean']
prepare_numeric = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
scaled_numeric = prepare_numeric.fit_transform(df_train[numeric])

kmeans = KMeans(n_clusters=2, random_state=1)
cluster = kmeans.fit_predict(scaled_numeric)

df_clusters = df_train.assign(cluster=cluster)
by_cluster = df_clusters.groupby('cluster')
summary = by_cluster.agg(patients=('died', 'size'), died=('died', 'mean'))
print(summary.round(3))

**» Make (stretch)** — **How many components?** Fit a `PCA` to `scaled_numeric` from the cell above, with no `n_components`, so it keeps all seven. **How many components hold 80% of the variation?** In the comment: what does that say about how much the seven columns overlap?

<details><summary><b>Hint</b></summary>

`pca.explained_variance_ratio_.cumsum()` adds the shares up, one component at a time. The heart-rate cell in section 5 has the pattern.

</details>

In [ ]:
# Components for 80% of the variation, and what that says about the seven columns:

# Your code here

## Regroup

Stop where you are: we come back together for the Closing.

If you didn't finish, finish after the session: Parts 1 to 4 are the core, and the stretch tasks are optional.

Finished early? Help a neighbour.

# 7. Closing
1. A forest, boosting, a network, and logistic regression with the distances: four very different models, and on these folds all four reach about 0.81. **What limits them, and what would you change to beat it: the model, or the table?**
2. You have now met five kinds of model. **For a new clinical table, which would you try first, and what would make you move on from it?**

## Wrap up
*   One deep tree learns its training patients' noise: high variance. **Bagging** averages many trees grown on bootstrap samples; a **random forest** also gives each split a random subset of features
*   **Boosting** adds small trees one after another, each correcting the last: it reduces bias. Tune the learning rate and the number of trees together
*   Resampling a rare outcome does not improve the AUC: it moves the threshold, and breaks the meaning of the risks. Choose a threshold instead, and if you resample, do it inside the pipeline
*   Clustering and PCA learn from the features alone. Scale first, and treat what they find as a question, not an answer
*   MIMIC data stay in your notebooks: do not share outputs, and do not paste rows into AI assistants

## Before the next session
1. **Finish the practice**, Parts 1 to 4, and check your decisions against the solutions notebook once it is released.
2. **Optional:** an open-data challenge on the Wisconsin breast cancer data, with nothing to submit: build the best classifier you can, then cluster the tumours and reduce their features with PCA: [Open in Colab](https://colab.research.google.com/github/stuartbowyer/sdsai-lecture-notes/blob/main/notebooks/exercises/Optional07_Breast_Cancer_Challenge.ipynb).
3. Reading: how gradient-boosted trees work, [xgboost.readthedocs.io/en/stable/tutorials/model.html](https://xgboost.readthedocs.io/en/stable/tutorials/model.html), and scikit-learn's guide to ensembles, [scikit-learn.org/stable/modules/ensemble.html](https://scikit-learn.org/stable/modules/ensemble.html); and why resampling harms a risk model, van den Goorbergh et al., [The harm of class imbalance corrections for risk prediction models](https://doi.org/10.1093/jamia/ocac093), *JAMIA* 2022. Optional: the original SMOTE paper, [arxiv.org/pdf/1106.1813](https://arxiv.org/pdf/1106.1813), and clustering for clinical phenotypes in heart failure, [link.springer.com/article/10.1007/s11897-023-00615-z](https://link.springer.com/article/10.1007/s11897-023-00615-z).

# Self-study appendices

## Bias and variance
Two ways for a model to be wrong on new patients:

*   **Bias:** wrong in the same way whatever patients it learns from, because it is too simple. Logistic regression's straight line through Session 6's temperature U was wrong at both ends, every time. Also called **underfitting**
*   **Variance:** a different answer from each set of training patients, because it is flexible enough to learn their noise. Also called **overfitting**

A shallow tree has high bias; a deep tree has high variance. The two **ensemble** methods today each attack one of them.

## AdaBoost: an earlier form of boosting
AdaBoost gives the rows each model got wrong more weight in the next model's training, and combines the models by a weighted vote. Gradient boosting, used by scikit-learn's `HistGradientBoostingClassifier`, instead fits each tree to every patient's error.

<img src="https://upload.wikimedia.org/wikipedia/commons/b/b5/Ensemble_Boosting.svg" alt="Boosting: models trained in sequence, each on the errors of the last" style="height: 400px; display: block; margin: 0 auto;">

By <a href="//commons.wikimedia.org/wiki/User:Sirakorn" title="User:Sirakorn">Sirakorn</a> - <span class="int-own-work" lang="en">Own work</span>, <a href="https://creativecommons.org/licenses/by-sa/4.0" title="Creative Commons Attribution-Share Alike 4.0">CC BY-SA 4.0</a>, <a href="https://commons.wikimedia.org/w/index.php?curid=85888769">Link</a>

## What the learning rate does
The average predicted risk of the test patients who died and who survived, as trees are added. Every model starts from the training death rate.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

died = y_test.to_numpy() == 1
rates = [0.01, 0.1, 1.0]

fig, axes = plt.subplots(1, len(rates), figsize=(12, 3.4), sharey=True)
for ax, rate in zip(axes, rates):
    model_rate = HistGradientBoostingClassifier(learning_rate=rate, random_state=0)
    model_rate.fit(X_train, y_train)
    # Average risk of each group after each tree, starting from the training death rate
    risk_died = [y_train.mean()]
    risk_survived = [y_train.mean()]
    for prob_stage in model_rate.staged_predict_proba(X_test):
        risk_died.append(prob_stage[died, 1].mean())
        risk_survived.append(prob_stage[~died, 1].mean())
    ax.plot(risk_died, color='#c0392b', linewidth=2, label='Test patients who died')
    ax.plot(risk_survived, color='#0000cd', linewidth=2, label='Test patients who survived')
    ax.set_title(f'learning_rate = {rate}', fontsize=10)
    ax.set_xlabel('Trees added')
    for side in ['top', 'right']:
        ax.spines[side].set_visible(False)
axes[0].set_ylabel('Average predicted risk')
axes[0].legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

## Introduction to Ensemble Methods
*   You probably have noticed that different models have different advantages and disadvantages
*   i.e. sometimes they work well, others they do not
*   Ensemble methods combine models together to improve overall performance by ...
    *   Improving accuracy
    *   Improving stability
    *   Reducing error

**How would you combine models together to optimise their group performance?**

## Oversampling
*   Add rows to the rarer class, here the deaths, in the training rows only:
*   **Random oversampling:** randomly duplicating entries from the minority class
*   **Synthetic Minority Oversampling Technique (SMOTE):** generating new synthetic samples in the minority class by interpolating between existing observations

### Random oversampling
Copies of the rarer class's rows, drawn at random, until the classes match. A flexible model can memorise the copies.

![Random Oversampling](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-06_Random-Oversampling.svg)

### SMOTE
New rows made by interpolating between a row of the rarer class and one of its near neighbours. No exact copies, but problems remain with:

*   Categories, such as admission type, where a point between two patients means nothing
*   Noise: an unusual row breeds more unusual rows
*   Many features, where "near neighbour" means little

![SMOTE](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-06_SMOTE-Oversampling.svg)

### Undersampling
*   Involves dropping observations in the majority class by ...
*   **Random undersampling:** randomly removing entries from the majority class
*   **Tomek links:** removes entries from the majority class that are close to the minority class (i.e. suspected noise)
*   **NearMiss:** removes entries from the majority class that are far from the minority class (i.e. easy classifications)


![Undersampling](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-06_Undersampling.svg)

### Using `imbalanced-learn`
*   Python has a package (parallelling `scikit-learn`) for addressing imbalanced datasets
    *    [Oversampling methods](https://imbalanced-learn.org/stable/references/over_sampling.html)
    *    [Undersampling methods](https://imbalanced-learn.org/stable/references/under_sampling.html)

`RandomOverSampler` on its own, outside a pipeline: the counts of each class before and after.

In [ ]:
from imblearn.over_sampling import RandomOverSampler

X = df_train[['age', 'heart_rate_mean', 'sbp_min', 'glucose_max']]
Y = df_train['died']

# Create the random oversampler
ros = RandomOverSampler(random_state=1)

# Apply it to our dataset
X_res, Y_res = ros.fit_resample(X, Y)

print("Previous value counts: ", Y.value_counts().to_list())
print("Resampled value counts:", Y_res.value_counts().to_list())

## Which features does the forest use?
**Permutation importance** shuffles one column of the test rows at a time, breaking its link with the outcome, and measures how far the AUC falls. A large fall: the model relies on that column. It describes the model, not the patients: two columns that carry the same information share the credit, and each can look unimportant.

In [ ]:
from sklearn.inspection import permutation_importance

importance = permutation_importance(forest, X_test, y_test, scoring='roc_auc', n_repeats=5, random_state=0)
fall = pd.Series(importance.importances_mean, index=X_test.columns)
fall = fall.sort_values(ascending=False)
top = fall.head(10)
top.round(3)

## Optional: a supervised learning challenge on open data
[Colab notebook](https://colab.research.google.com/github/stuartbowyer/sdsai-lecture-notes/blob/main/notebooks/exercises/Optional07_Breast_Cancer_Challenge.ipynb)

#### Properties of K-Means Clustering
| **Advantages**                              | **Disadvantages**                                       |
|---------------------------------------------|-------------------------------------------------------|
| Simple and easy to implement                | Requires the number of clusters (\(k\)) to be predefined |
| Computationally efficient for large datasets| Sensitive to the initial choice of centroids          |
| Works well with spherical cluster shapes    | Struggles with clusters of varying sizes or densities |
| Easily interpretable results                | Prone to convergence to local minima                 |
| Scales well to higher-dimensional data      | Affected by outliers, which can distort cluster centroids |

### Feature Selection
*   **What:** completely removing one or more features that do not contribute 'useful information'
*   As covered in previous reading for supervised learning
*   **How:** various methods available that remove features with:
    *   Low variance (i.e. features that do not change much)
    *   High correlation (i.e. two features represent the same physical characteristic)
    *   High mutual information (i.e. two features with high dependency or high shared information)

### PCA on the three heart-rate columns, in pictures
The three columns in three dimensions, and the same patients on the first two components. The plots show individual patients, so they appear only in your own notebook.

In [ ]:
# This cell's code uses plain arrays, not tables
sklearn.set_config(transform_output='default')

from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Select the two columns for analysis
X = df_model[['heart_rate_max', 'heart_rate_mean', 'heart_rate_min']].dropna().to_numpy()

# Create and fit the PCA
pca = PCA(n_components=2, random_state=1)
pca.fit(X)

# Print the Explained Variance Ratio
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")

# Transform the data
X_trans = pca.transform(X)

# Plot the Full Heart Rate Data
fig = plt.figure(figsize=(18, 8))
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(X[:, 0], X[:, 1], X[:, 2], alpha=0.2)
ax1.set_xlabel("Max HR")
ax1.set_ylabel("Mean HR")
ax1.set_zlabel("Min HR")

# Plot the Dimension Reduced Data
ax2 = fig.add_subplot(122)
ax2.scatter(X_trans[:, 0], X_trans[:, 1])
ax2.set_xlabel('PC1')
ax2.set_ylabel('PC2')
plt.show()
sklearn.set_config(transform_output='pandas')

## Optional: clustering the Wisconsin breast cancer data
[Colab notebook](https://colab.research.google.com/github/stuartbowyer/sdsai-lecture-notes/blob/main/notebooks/exercises/Optional07_Breast_Cancer_Challenge.ipynb)

## Optional: PCA on the Wisconsin breast cancer data
[Colab notebook](https://colab.research.google.com/github/stuartbowyer/sdsai-lecture-notes/blob/main/notebooks/exercises/Optional07_Breast_Cancer_Challenge.ipynb)

## Model choice and the data
*   You now have lots of tools to address supervised and unsupervised machine learning problems; however, experience and reading up on the nuances of each method is the difference between using them well
*   Data preparation is critically important to model performance
*   Consider the bias-variance trade offs
*   REMEMBER... if your model does not have the key features of the system/disease/procedure/etc. you are trying to predict, no matter how complex you make your model or how many samples you collect, it will never improve its performance

<img src="https://assets.cureus.com/uploads/figure/file/292667/article_river_6a398820513311ecb45235ba7f0fde7e-LRvsNN_plot4.png" alt="Neural networks against logistic regression across healthcare studies" style="height: 400px; display: block; margin: 0 auto;">

Issitt R W, Cortina-Borja M, Bryant W, et al. (February 21, 2022) Classification Performance of Neural Networks Versus Logistic Regression Models: Evidence From Healthcare Practice. Cureus 14(2): e22443. doi:10.7759/cureus.22443

## Self Study

### New Material
*   Detail on Gradient Boosting / XGboost - [xgboost.readthedocs.io/en/stable/tutorials/model.html](https://xgboost.readthedocs.io/en/stable/tutorials/model.html)
*   Silhouette scoring of cluster separation - [scikit-learn.org/1.5/auto_examples/cluster/plot_kmeans_silhouette_analysis.html](https://scikit-learn.org/1.5/auto_examples/cluster/plot_kmeans_silhouette_analysis.html)
*   t-SNE (t-distributed Stochastic Neighbor Embedding) tutorial - [www.datacamp.com/tutorial/introduction-t-sne](https://www.datacamp.com/tutorial/introduction-t-sne)

### Consolidation Reading
*   Original SMOTE Oversampling Method - [arxiv.org/pdf/1106.1813](https://arxiv.org/pdf/1106.1813)
*   Boosting and Bagging Paper - [www.d.umn.edu/~rmaclin/cs5751/notes/opitz-jair99.pdf](https://www.d.umn.edu/~rmaclin/cs5751/notes/opitz-jair99.pdf)
*   Survey article on using clustering for discovering clinical phenotypes in cardiac disease - [link.springer.com/article/10.1007/s11897-023-00615-z](https://link.springer.com/article/10.1007/s11897-023-00615-z)
*   `scikit-learn`'s high-level overview of clustering methods - [scikit-learn.org/1.5/modules/clustering.html](https://scikit-learn.org/1.5/modules/clustering.html)